LogisticRegressor: a simplt custom built class implementation for the perceptron logic.

src.shared imports: \
numpy as np \
pandas as pd \
matplotlib.pyplot as plt \
utils like load_dataset, calculating metrics, etc.

**NOTE** \
I know logistic regression is standardly used for classification. \
The target variable in this dataset provides continuous probabilities. \
This implementation intentionally predicts these exact probability values rather than converting them to binary labels.

In [16]:
import sys
from pathlib import Path

project_root = str(Path.cwd().parents[1])

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.regression.logistic import LogisticRegressor
from src.shared import *

I chose a simple dataset suitable for probability calculation. \
The Floods dataset fits these criteria well and is perfect for demonstrating the logistic regressor.

In [17]:
# --- 1. Load Data ---
df = load_dataset('floods')


# --- 2. Data Cleaning ---
df = clean_dataframe(df)

print(df.head(), '\n')


# --- 3. Analysis ---
# Correlation Matrix (Simple 1-to-1 relationship)
correlations = df.corr()['flood_probability'].sort_values(ascending=False).drop('flood_probability')
print("A table of features correlations with flood probability:")
print(correlations)

# We can see there is no particular feature that has a very high correlation with flood probability. 
# This means we will need to use as much features as we can to predict flood probability with the most accuracy.

   monsoon_intensity  topography_drainage  river_management  deforestation  \
0                  3                    8                 6              6   
1                  8                    4                 5              7   
2                  3                   10                 4              1   
3                  4                    4                 2              7   
4                  3                    7                 5              2   

   urbanization  climate_change  dams_quality  siltation  \
0             4               4             6          2   
1             7               9             1          5   
2             7               5             4          7   
3             3               4             1          4   
4             5               8             5          2   

   agricultural_practices  encroachments  ...  drainage_systems  \
0                       3              2  ...                10   
1                       5           

In [20]:
# --- 1. Select Features & Target ---
# using all features for prediction (we sadly can't plot a 2D or 3D graph because of this)
X = df.drop('flood_probability', axis=1).values
y = df['flood_probability'].values


# --- 2. Split Data for training & testing ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=None)


# Normalize features to 0-1 range for better performance
train_min = X_train.min(axis=0)
train_range = X_train.max(axis=0) - train_min + 1e-15

X_train = (X_train - train_min) / train_range
X_test = (X_test - train_min) / train_range


# --- 3. Training ---
print(f"Training on {len(X_train)} samples")

model = LogisticRegressor()

# Small inputs (0-1) yield small gradients, so we use a high Learning Rate for fast convergence.
model.fit(X_train, y_train, learning_rate=1, n_epochs=2000, show_progress=True)


predictions = model.predict(X_test)

mae = Metrics.mae(y_test, predictions)
r2 = Metrics.r2(y_test, predictions)

print(f"Mean Absolute Error: {mae:.4f}")
print(f"R-squared Score: {r2:.4f}")

Training on 40000 samples
Epoch 0 - Loss: 0.6932
Epoch 200 - Loss: 0.6908
Epoch 400 - Loss: 0.6895
Epoch 600 - Loss: 0.6889
Epoch 800 - Loss: 0.6885
Epoch 1000 - Loss: 0.6883
Epoch 1200 - Loss: 0.6882
Epoch 1400 - Loss: 0.6882
Epoch 1600 - Loss: 0.6882
Epoch 1800 - Loss: 0.6881
Mean Absolute Error: 0.0017
R-squared Score: 0.9980
